<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

This exercise covers two aspects:
* In tasks 1-6 you will implement mechanisms that allow training deeper models. After doing each of the tasks you can look at the plots and check how your changes impact gradients of network layers.
* In task 7 you will implement a convnet using [conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) and try to reach 99% accuracy on MNIST.

For this notebook, running on CPU is sufficient: tasks 1–5. run under a minute, 6. under two minutes, and 7. up to 5 minutes.

Tasks:
1. Check that the given implementation reaches 95% test accuracy for
   the architecture `input-64-64-10` in 8 epochs.
2. Modify `Linear.reset_parameters()` to improve initialization and
   check that the network learns much faster and reaches over 97% test accuracy.
   A good basic initialization scheme is so-called *Glorot initialization*.
   For a linear layer with $n_{in}$ and  $n_{out}$ input and output features,
   it samples each weight from a normal distribution with $0$ mean and a standard deviation of $\sqrt{\frac{2}{n_{in}+n_{out}}}$.  
   Check how that changes the distribution of gradients at the first epoch.
3. Check, that with proper initialization we can train the architecture
   `input-64-64-64-64-64-64-64-10` (seven hidden layers of `64`), for e.g. 3 epochs,
   while with the original, bad initialization, it does not even get off the ground.
4. Implement Dropout as a PyTorch module (without using torch.nn.Dropout).
   Use `p=0.25` as the zeroing probability.
   Use Dropout after activations of each layer, except the last.
5. Check that with 10 hidden layers (64 units each), even with proper initialization,
   but with dropout added, the network has a hard time to start learning.
6. Implement batch normalization (without using torch.nn.BatchNorm).
    * compute batch mean and variance
    * add new variables beta and gamma
    * check that the network learns even for 10 hidden layers.
    * check how gradients change in comparison to network without batch norm.
   
   In pseudo-code:
    ```
		if training:
		    var, mean = torch.var_mean(x, dim=/batch/)
		    update running_mean, running_var
		else:
			mean, var = running_mean, running_var
		x = (x - mean) / sqrt(var + eps)
		return x * gamma + beta
    ```

7. Design and implement a simple convolutional network and achieve 99% test accuracy (in up to 10 epochs, 5 should be enough). The architecture is up to you, but even a few convolutional layers should be enough.
You may use `torch.nn.Conv2d, BatchNorm2d, MaxPool2d, ReLU, Dropout2d, Linear, Flatten, BatchNorm1d`.
Don't make it too generic, just try a few fixed layers.

In [1]:
import math
import sys

import numpy as np
import PIL.Image
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from plotly.subplots import make_subplots
from torch import Tensor
from torch.nn import init

if 'google.colab' in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
# @title Visualize gradients

class GradientVisualizer:
    def __init__(self, net: nn.Module, num_epochs: int) -> None:
        self.num_epochs = num_epochs
        self.linear_layers = self.get_linear_layers(net)

        self.grad_to_weight_fig = None
        self.grads_in_layers_fig = None
        self.grads_at_epochs_fig = None
        self.init_figures()

    @staticmethod
    def get_linear_layers(net):
        linear_layers = []
        for field in net.modules():
            if isinstance(field, nn.Linear | Linear):
                linear_layers.append(field)
        assert linear_layers, "No linear layers found. Forgot to use Sequential or ModuleList?"
        return linear_layers

    def get_epochs_for_one_layer(self):
        """
        We want to show gradient distributions from up to 7 selected epochs
        for one linear layer.
        """
        if self.num_epochs < 7:
            return list(range(self.num_epochs))
        else:
            return torch.linspace(0, self.num_epochs - 1, 7).int().tolist()

    def get_three_epochs(self):
        """
        We want to show gradients distributions from all layers at each of
        three epochs: first, middle and last.
        """
        return [0, self.num_epochs // 2, self.num_epochs - 1]

    def rgb_to_rgba(self, rgb_color, epoch):
        """
        Value of epoch parameter determines how transparent color should be
        in comparison to others.
        Colors for earlier epochs should be more transparent/less visible.
        """
        return f'rgba{rgb_color[3:-1]},{0.6 * (epoch + 1) / self.num_epochs + 0.15})'

    def init_figures(self):
        # Initialize figure with gradient to weight ratio plot
        fig = go.Figure()
        fig.update_layout(
            title='Gradient stddev to weight stddev ratio', title_x=0.5,
            xaxis_title='Epoch',
            yaxis_title='Gradient to weight ratio (log scale)',
            height=400, width=1500, margin=dict(b=10, t=60)
        )
        fig.update_yaxes(type='log')
        for i in range(len(self.linear_layers)):
            fig.add_trace(go.Scatter(
                x=[], y=[],
                mode='lines+markers', name=f'Linear layer {i}'
            ))

        self.grad_to_weight_fig = go.FigureWidget(fig)
        display(self.grad_to_weight_fig)

        # Initialize figure visualizing gradient distributions in layers
        chosen_layers = [0, len(self.linear_layers) // 2, len(self.linear_layers) - 1]
        num_rows = (len(chosen_layers) - 1) // 3 + 1
        fig = make_subplots(
            rows=num_rows, cols=3,
            subplot_titles=[f'Linear layer {i}' for i in chosen_layers],
            vertical_spacing=0.2 / num_rows
        )
        fig.update_layout(
            title='Gradient distributions by epoch (for three chosen layers)', title_x=0.5,
            height=num_rows * 400, width=1500, margin=dict(b=10, t=60)
        )

        colors, _ = px.colors.convert_colors_to_same_type(2 * px.colors.qualitative.Plotly)
        for i, layer_num in enumerate(chosen_layers):
            row = i // 3 + 1
            col = i % 3 + 1
            fig.update_xaxes(
                title_text='Gradient value', range=(-0.1, 0.1), row=row, col=col
            )
            fig.update_yaxes(
                title_text='Density (log scale)', type='log', row=row, col=col
            )

            # Create empty traces and update them later with actual gradient distributions.
            # Unfortunately, we cannot add new traces dynamically because Colab has problem
            # with widgets from plotly (traces added dynamically are rendered twice).
            for epoch in self.get_epochs_for_one_layer():
                fig.add_trace(
                    go.Scatter(
                        mode='lines', name=f'Epoch {epoch + 1}',
                        line=dict(color=self.rgb_to_rgba(colors[layer_num], epoch)),
                        legendgroup=layer_num
                    ),
                    row=row, col=col
                )

        self.grads_in_layers_fig = go.FigureWidget(fig)
        display(self.grads_in_layers_fig)

        # Initialize figure comparing gradient distributions between layers at the
        # first, middle and last epoch
        selected_epochs_indices = self.get_three_epochs()
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=[f'Epoch {epoch + 1}' for epoch in selected_epochs_indices]
        )
        fig.update_layout(
            title='Gradient distributions by layers (for three chosen epochs)', title_x=0.5,
            height=400, width=1500, margin=dict(b=10, t=60)
        )

        for col, epoch in enumerate(selected_epochs_indices, 1):
            fig.update_yaxes(title_text='Density (log scale)', type='log', row=1, col=col)
            fig.update_xaxes(
                title_text='Gradient value',
                range=(-0.05, 0.05) if epoch != 0 else (-1, 1),
                row=1, col=col
            )

            # Create empty traces and update them later with actual gradient distributions.
            for layer_num in range(len(self.linear_layers)):
                fig.append_trace(
                    go.Scatter(
                        mode='lines', name=f'Linear layer {layer_num}',
                        line=dict(color=colors[layer_num]), showlegend=(col == 1)
                    ),
                    row=1, col=col
                )

        self.grads_at_epochs_fig = go.FigureWidget(fig)
        display(self.grads_at_epochs_fig)

    def visualize_gradients(self, lr, epoch, batch_idx):
        # It is enough to use gradients calculated for the first batch.
        if batch_idx != 0:
            return

        epoch_grads = []
        epoch_grad_to_weight_ratios = []
        for layer in self.linear_layers:
            epoch_grads.append(layer.weight.grad.flatten().detach())
            epoch_grad_to_weight_ratios.append(
                (lr * layer.weight.grad.std() / layer.weight.std()).item()
            )

        # Update figure with gradient to weight ratio plot
        assert self.grad_to_weight_fig is not None
        for i, grad_to_weight_ratio in enumerate(epoch_grad_to_weight_ratios):
            x = self.grad_to_weight_fig.data[i].x
            next_x_val = x[-1] + 1 if x else 1
            self.grad_to_weight_fig.data[i].x += (next_x_val, )
            self.grad_to_weight_fig.data[i].y += (grad_to_weight_ratio, )

        # Update figure visualizing gradient distributions in layers
        assert self.grads_in_layers_fig is not None
        chosen_layers = [0, len(self.linear_layers) // 2, len(self.linear_layers) - 1]
        selected_epochs = self.get_epochs_for_one_layer()
        if epoch in selected_epochs:
            epoch_idx = selected_epochs.index(epoch)
            for layer_num, layer_grad in enumerate(epoch_grads):
                try:
                    layer_id = chosen_layers.index(layer_num)
                except ValueError:
                    continue
                trace_idx = layer_id * len(selected_epochs) + epoch_idx
                try:
                    hy, hx = torch.histogram(layer_grad, bins=50, density=True)
                except RuntimeError:
                    continue
                hy = hy / max(hy) + 0.001
                self.grads_in_layers_fig.data[trace_idx].x = hx[:-1].tolist()
                self.grads_in_layers_fig.data[trace_idx].y = hy.tolist()


        # Update figure visualizing gradient distributions at epochs
        assert self.grads_at_epochs_fig is not None
        selected_epochs = self.get_three_epochs()
        if epoch in selected_epochs:
            epoch_idx = selected_epochs.index(epoch)
            for layer_num, layer_grad in enumerate(epoch_grads):
                trace_idx = epoch_idx * len(self.linear_layers) + layer_num
                try:
                    hy, hx = torch.histogram(layer_grad, bins=50, density=True)
                except RuntimeError:
                    continue
                hy = hy / max(hy) + 0.001
                self.grads_at_epochs_fig.data[trace_idx].x = hx[:-1].tolist()
                self.grads_at_epochs_fig.data[trace_idx].y = hy.tolist()

In [3]:
#Static version of the GradientVisualizer plots.
#The interactive FigureWidgets do not render in VS Code and are not saved in the notebook,
#so after training we copy them into normal plotly figures (these render and are saved).
def show_static(gv: GradientVisualizer) -> None:
    for fig in (gv.grad_to_weight_fig, gv.grads_in_layers_fig, gv.grads_at_epochs_fig):
        go.Figure(fig).show()

In [4]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, glorot_init: bool) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(Tensor(out_features, in_features))
        self.bias = nn.Parameter(Tensor(out_features))
        self.glorot_init = glorot_init
        self.reset_parameters()

    def reset_parameters(self) -> None:
        if self.glorot_init:
            std = math.sqrt(2/ (self.in_features + self.out_features))
            self.weight.data.normal_(mean=0, std=std)
        else:
            self.weight.data.normal_(mean=0, std=0.25)
        init.zeros_(self.bias)

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (*B, in_features)
        Output: shape (*B, out_features)
        """
        r = x.matmul(self.weight.T)
        r += self.bias
        return r


class Net(nn.Module):
    def __init__(self, glorot_init: bool = False) -> None:
        super().__init__()
        self.fc1 = Linear(784, 64, glorot_init=glorot_init)
        self.fc2 = Linear(64, 64, glorot_init=glorot_init)
        self.fc3 = Linear(64, 10, glorot_init=glorot_init)

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, H=28, W=28)
        Output: shape (B, num_classes=10)
        """
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [5]:
class MnistTrainer:
    def __init__(self, batch_size: int = 128) -> None:
        transform = transforms.Compose([transforms.ToTensor()])
        self.trainset = torchvision.datasets.MNIST(
            root='./data', download=True, train=True, transform=transform
        )
        self.trainloader = torch.utils.data.DataLoader(
            self.trainset, batch_size=batch_size, shuffle=True, num_workers=2
        )
        self.testset = torchvision.datasets.MNIST(
            root='./data', train=False, download=True, transform=transform
        )
        self.testloader = torch.utils.data.DataLoader(
            self.testset, batch_size=1000, shuffle=False, num_workers=2
        )

    def train(
        self,
        net: nn.Module,
        gradient_visualizer: GradientVisualizer | None = None,
        epochs: int = 20,
        lr: float = 0.05,
        momentum: float = 0.9
    ) -> None:
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum)
        log_freq = 100

        for epoch in range(epochs):
            net.train()
            running_loss = 0.0
            for i, data in enumerate(self.trainloader):
                inputs, labels = data
                optimizer.zero_grad()

                outputs = net(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                if gradient_visualizer is not None:
                    gradient_visualizer.visualize_gradients(lr, epoch, i)
                optimizer.step()

                running_loss += loss.item()
                if i % log_freq == log_freq - 1:
                    running_loss /= log_freq
                    print(f"epoch {epoch + 1:>5}, batch {i + 1:>5}, loss {running_loss:.3f}")
                    running_loss = 0.0
            self.test(net)

    def test(self, net: nn.Module) -> list[float]:
        net.eval()
        correct = 0
        total = 0
        all_gt_logits = []
        with torch.no_grad():
            for data in self.testloader:
                images, labels = data
                outputs = net(images)
                gt_logits = torch.gather(outputs, dim=1, index=labels.unsqueeze(-1))
                all_gt_logits.extend(float(logit.item()) for logit in gt_logits)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f"Accuracy of the network on the {total} test images: {correct / total:.2%}")
        return all_gt_logits

    def show_worst_misclassified(self, net: nn.Module, k: int = 10) -> None:
        all_gt_logits = self.test(net)
        _, indices = torch.topk(-torch.tensor(all_gt_logits), k)
        imgs = []
        labels = []
        for i in indices:
            img, label = self.testset[int(i.item())]
            imgs.append(img)
            labels.append(label)
        image = PIL.Image.fromarray(
            torch.cat(imgs, dim=-1).mul(255).to(torch.uint8).permute(1, 2, 0).numpy().squeeze()
        )
        display(image.resize((5 * image.width, 5 * image.height), PIL.Image.Resampling.NEAREST))
        print(labels)

# Task 1 & 2: Baseline and Glorot initialization

In [6]:
epochs = 20

net = Net(glorot_init=True)
gradient_visualizer = GradientVisualizer(net, epochs)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '2664be38-17b4-4b05-ae36-489c0681d4f2',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': 'a3434fbe-8efe-45fe-8f69-34b739c62eb1',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '4f3502c0-ae96-44ff-b616-9182a786f399',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 't': 60},
               'template': '...',
               'title': {'text': 'Gradient stddev to weight stddev ratio', 'x': 0.5},
               'width': 1500,
               'xaxis': {'title': {'text': 'Epoch'}},
               'yaxis': {'title': {'text': 'G

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.18)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '6a222e0d-6f90-400a-9821-639100bd1a71',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.27)'},
              'mode': 'lines',
              'name': 'Epoch 4',
              'type': 'scatter',
              'uid': '951423fe-0218-486c-afb6-d7230fbaf25c',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.36)'},
              'mode': 'lines',
              'name': 'Epoch 7',
              'type': 'scatter',
              'uid': 'cbc5f5a7-599a-4a30-898c-20ef0bb337f7',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'c2639d39-b8bf-4b87-ba2e-16c6607cdf3d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '761aada8-6076-4b4d-9b25-e1ca16c1b75b',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'b6ce6b31-5257-4037-ac25-190ccaf88362',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(99, 110, 250)'},
   

KeyError: 'uid'

KeyError: 'uid'

In [7]:
trainer = MnistTrainer(batch_size=128)
trainer.train(net, gradient_visualizer, epochs=epochs)
show_static(gradient_visualizer)

epoch     1, batch   100, loss 0.695
epoch     1, batch   200, loss 0.302
epoch     1, batch   300, loss 0.234
epoch     1, batch   400, loss 0.189
Accuracy of the network on the 10000 test images: 95.36%
epoch     2, batch   100, loss 0.149
epoch     2, batch   200, loss 0.146
epoch     2, batch   300, loss 0.138
epoch     2, batch   400, loss 0.116
Accuracy of the network on the 10000 test images: 95.82%
epoch     3, batch   100, loss 0.099
epoch     3, batch   200, loss 0.100
epoch     3, batch   300, loss 0.096
epoch     3, batch   400, loss 0.096
Accuracy of the network on the 10000 test images: 96.73%
epoch     4, batch   100, loss 0.077
epoch     4, batch   200, loss 0.076
epoch     4, batch   300, loss 0.077
epoch     4, batch   400, loss 0.082
Accuracy of the network on the 10000 test images: 96.98%
epoch     5, batch   100, loss 0.063
epoch     5, batch   200, loss 0.061
epoch     5, batch   300, loss 0.067
epoch     5, batch   400, loss 0.073
Accuracy of the network on the 1

In [8]:
#Task 1 baseline: the given implementation (original init, std=0.25) for 8 epochs
epochs = 8
baseline_net = Net()
trainer.train(baseline_net, None, epochs=epochs)

epoch     1, batch   100, loss 0.916
epoch     1, batch   200, loss 0.334
epoch     1, batch   300, loss 0.288
epoch     1, batch   400, loss 0.238
Accuracy of the network on the 10000 test images: 93.42%
epoch     2, batch   100, loss 0.194
epoch     2, batch   200, loss 0.183
epoch     2, batch   300, loss 0.178
epoch     2, batch   400, loss 0.161
Accuracy of the network on the 10000 test images: 95.47%
epoch     3, batch   100, loss 0.129
epoch     3, batch   200, loss 0.137
epoch     3, batch   300, loss 0.144
epoch     3, batch   400, loss 0.140
Accuracy of the network on the 10000 test images: 95.84%
epoch     4, batch   100, loss 0.107
epoch     4, batch   200, loss 0.110
epoch     4, batch   300, loss 0.124
epoch     4, batch   400, loss 0.117
Accuracy of the network on the 10000 test images: 96.07%
epoch     5, batch   100, loss 0.093
epoch     5, batch   200, loss 0.098
epoch     5, batch   300, loss 0.096
epoch     5, batch   400, loss 0.111
Accuracy of the network on the 1

# Task 3, 4 & 6: Deeper network, Dropout and BatchNorm

In [9]:
#Dropout
class Dropout(nn.Module):
    def __init__(self,p:float = 0.25) -> None: 
        super().__init__()
        self.p = p

    def forward(self,x:Tensor) -> Tensor:
        if self.training:
            mask = (torch.rand_like(x)>self.p).float()
            return x * mask / (1-self.p)
        return x

In [10]:
#BatchNorm
class BatchNorm(nn.Module):
    def __init__(self, num_features: int, momentum: float = 0.1, eps: float = 1e-5) -> None:
        super().__init__()
        self.momentum = momentum
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x: Tensor) -> Tensor:
        if self.training:
            var, mean = torch.var_mean(x, dim=0, unbiased=False)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            mean, var = self.running_mean, self.running_var
        x = (x - mean) / torch.sqrt(var + self.eps)  
        return self.gamma * x + self.beta

In [11]:
class DeepNet(nn.Module):
    def __init__(self,glorot_init:bool = False,
                num_hidden: int = 7,
                dropout: bool = False,
                batch_norm: bool = False) -> None:
        
        super().__init__()
        sizes = [784]
        sizes.extend([64]*num_hidden)
        sizes.append(10)
        self.layers = nn.ModuleList()
        for a,b in zip(sizes[:-1],sizes[1:]):
            self.layers.append(Linear(a,b,glorot_init=glorot_init))
        self.use_dropout = dropout
        self.drop = Dropout(p=0.25)
        self.use_bn = batch_norm
        self.bns = nn.ModuleList()
        if self.use_bn:
            for _ in range(num_hidden):
                self.bns.append(BatchNorm(64))

    def forward(self,x:Tensor) -> Tensor:
        x = x.view(-1,784)
        for i,layer in enumerate(self.layers[:-1]):
            x = layer(x)
            if self.use_bn:
                x = self.bns[i](x)
            x = F.relu(x)
            if self.use_dropout:
                x = self.drop(x)
        x = self.layers[-1](x)
        return x

In [12]:
#Training the deepnet [784, 64, 64, 64, 64, 64, 64, 64, 10] 
# (glorot initialization)
epochs = 3
deepnet = DeepNet(glorot_init=True)
gradient_visualizer = GradientVisualizer(deepnet,epochs)
trainer.train(deepnet,gradient_visualizer,epochs=epochs)
show_static(gradient_visualizer)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': 'b1369cfa-98ab-4b63-baa2-e4ff9631f2c9',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '23580bf7-6333-46a3-a8df-b2445c4de3d2',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '52f9db45-d00b-43d4-a00c-81bb201d9c2a',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': 'd1a649aa-366a-4d9f-ba6d-9ace5085150c',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '2a2e8e4e-e74d-41ac-af47-225a0388c991',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': '59fb7529-c292-4a8c-8b8b-83fb1082632e',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '3537365e-6ff0-4707-b442-8217a6c6a16f',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '4',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '7a33ee12-0399-42bd-8d5b-f3821ab0ebea',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '968feb14-abe1-48a3-9cd1-b893b6dbf2ae',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'f667ac83-aeb5-4a61-a742-a4f376748f94',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

epoch     1, batch   100, loss 1.381
epoch     1, batch   200, loss 0.378
epoch     1, batch   300, loss 0.306
epoch     1, batch   400, loss 0.269
Accuracy of the network on the 10000 test images: 94.36%
epoch     2, batch   100, loss 0.175
epoch     2, batch   200, loss 0.176
epoch     2, batch   300, loss 0.182
epoch     2, batch   400, loss 0.151
Accuracy of the network on the 10000 test images: 95.56%
epoch     3, batch   100, loss 0.139
epoch     3, batch   200, loss 0.128
epoch     3, batch   300, loss 0.121
epoch     3, batch   400, loss 0.134
Accuracy of the network on the 10000 test images: 95.95%


In [13]:
#Training the same deepnet with standard initialization:
epochs = 3
deepnet = DeepNet(glorot_init=False)
gradient_visualizer = GradientVisualizer(deepnet,epochs)
trainer.train(deepnet,gradient_visualizer,epochs=epochs)
show_static(gradient_visualizer)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': 'eccf159f-b96f-4ca1-bd63-45d97890a4ac',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': 'd799a568-2381-4e7c-9c34-38e841b752bc',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '4f744697-bcc6-4246-92fd-b2fc719a0ec3',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': 'c2e712b0-924d-4991-97c7-e21c10092a99',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '394a14b7-ff47-4b31-aa27-e81f90b31fd3',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'b3b0f21f-6217-4c36-b36a-c0b4b59a15e1',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '6aa32a3d-d3ed-4325-a69d-14655255898c',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '4',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '9546b3d1-62d8-4da3-ab2b-5a5e0f34f67e',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '600fe90c-5cd7-4142-9603-bf21d90afad7',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'aeab5315-3f4d-46a5-b4d7-87520750c45b',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

epoch     1, batch   100, loss 3.136
epoch     1, batch   200, loss 1.564
epoch     1, batch   300, loss 1.305
epoch     1, batch   400, loss 1.022
Accuracy of the network on the 10000 test images: 72.98%
epoch     2, batch   100, loss 0.727
epoch     2, batch   200, loss 0.687
epoch     2, batch   300, loss 0.589
epoch     2, batch   400, loss 0.482
Accuracy of the network on the 10000 test images: 87.52%
epoch     3, batch   100, loss 0.435
epoch     3, batch   200, loss 0.409
epoch     3, batch   300, loss 0.373
epoch     3, batch   400, loss 0.379
Accuracy of the network on the 10000 test images: 90.39%


# Task 5: 10 hidden layers with dropout

In [14]:
#Training a deepnet with 10 hidden layers, glorot initialization and dropout
epochs = 3
deepnet = DeepNet(glorot_init=True, num_hidden=10, dropout=True)
gradient_visualizer = GradientVisualizer(deepnet,epochs)
trainer.train(deepnet,gradient_visualizer,epochs=epochs)
show_static(gradient_visualizer)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '010c45a5-4552-4fe2-acfa-90e6a4fcf5f3',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': 'bc2d20ec-fbdb-4392-860d-f4bdcf2eb408',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '14377408-fa58-4359-b7d2-07c8b335c8b5',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '452d1a20-5aaf-4108-a4f2-325abcd88670',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': 'd9f79a42-6505-44b8-9198-f58aa2ba5cb6',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': '08c529d5-a2ee-44e2-8be9-b8432fff74fa',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '2d775b61-3aa0-4736-bdf0-650806a5668a',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '5',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '3ba3be1e-0efe-4d46-b7cc-cf2d91287b04',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '598e0908-fff4-420c-bdf2-20f0cb39c44d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': '20e0658b-af37-42e1-ab9b-12470d15e097',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

epoch     1, batch   100, loss 2.153
epoch     1, batch   200, loss 1.773
epoch     1, batch   300, loss 1.659
epoch     1, batch   400, loss 1.540
Accuracy of the network on the 10000 test images: 44.49%
epoch     2, batch   100, loss 1.336
epoch     2, batch   200, loss 1.257
epoch     2, batch   300, loss 1.193
epoch     2, batch   400, loss 1.057
Accuracy of the network on the 10000 test images: 66.77%
epoch     3, batch   100, loss 1.054
epoch     3, batch   200, loss 1.032
epoch     3, batch   300, loss 1.012
epoch     3, batch   400, loss 0.974
Accuracy of the network on the 10000 test images: 68.96%


In [15]:
#Same deepnet (10 hidden layers, glorot initialization) without dropout, for comparison
epochs = 3
deepnet = DeepNet(glorot_init=True, num_hidden=10, dropout=False)
gradient_visualizer = GradientVisualizer(deepnet,epochs)
trainer.train(deepnet,gradient_visualizer,epochs=epochs)
show_static(gradient_visualizer)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': 'cc502934-6da8-4a52-a059-0869de7b067c',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': 'f5c29ce4-818b-4fe8-b89d-f8aac6ca05df',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '93028dea-bdfe-4fd5-862e-42928b33e321',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': 'f1f8da14-79fa-4e54-b574-c29bde717770',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '5468c865-7004-4cda-aece-f3245d7bff56',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'ec273c88-47b4-4e11-98e2-c177f3aeeae0',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '75d44f1d-8c04-4e21-b4a0-8a526803e205',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '5',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '5d361f76-1291-460b-b779-a5ab18b5d619',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': '52a51d05-7b24-43dd-8fb7-1c51f67ffa33',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'd15ba27a-3ef1-4800-ab9e-ac49ddd7abab',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

epoch     1, batch   100, loss 1.952
epoch     1, batch   200, loss 0.907
epoch     1, batch   300, loss 0.434
epoch     1, batch   400, loss 0.336
Accuracy of the network on the 10000 test images: 93.53%
epoch     2, batch   100, loss 0.256
epoch     2, batch   200, loss 0.262
epoch     2, batch   300, loss 0.230
epoch     2, batch   400, loss 0.202
Accuracy of the network on the 10000 test images: 94.58%
epoch     3, batch   100, loss 0.172
epoch     3, batch   200, loss 0.163
epoch     3, batch   300, loss 0.153
epoch     3, batch   400, loss 0.154
Accuracy of the network on the 10000 test images: 95.58%


# Task 6: Batch Normalization

In [16]:
#Deepnet with 10 hidden layers, glorot initialization, dropout and batch norm
epochs = 3
deepnet = DeepNet(glorot_init=True, num_hidden=10, dropout=True, batch_norm=True)
gradient_visualizer = GradientVisualizer(deepnet,epochs)
trainer.train(deepnet,gradient_visualizer,epochs=epochs)
show_static(gradient_visualizer)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': '57bcf712-282a-4622-85a1-7131691778f7',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': '092e418e-6385-4bd8-bd75-e715ffd0c614',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '55974af9-8cdc-49c4-bc8f-b2de291f0c3c',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '626f5471-d7ab-40fb-ba45-a7b00fec5cfc',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '42957546-0ca6-4fbd-8bae-2a2fd2376d22',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'bb45ebb7-ab71-4279-b57a-070a8f10b7f6',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': 'b38c3076-5088-4902-8497-340d6583949d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '5',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': '42c05823-b688-48d6-a8e4-29e21d6df6af',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'ba0116ec-2fe5-437d-8048-18d6125b7de2',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': '934161e7-bd42-4b96-9717-70eaeb02c97b',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

epoch     1, batch   100, loss 2.072
epoch     1, batch   200, loss 1.422
epoch     1, batch   300, loss 1.065
epoch     1, batch   400, loss 0.899
Accuracy of the network on the 10000 test images: 84.62%
epoch     2, batch   100, loss 0.757
epoch     2, batch   200, loss 0.704
epoch     2, batch   300, loss 0.641
epoch     2, batch   400, loss 0.641
Accuracy of the network on the 10000 test images: 93.09%
epoch     3, batch   100, loss 0.595
epoch     3, batch   200, loss 0.562
epoch     3, batch   300, loss 0.530
epoch     3, batch   400, loss 0.514
Accuracy of the network on the 10000 test images: 94.10%


In [17]:
#Deepnet with 10 hidden layers, glorot initialization and batch norm (no dropout), for comparison
epochs = 3
deepnet = DeepNet(glorot_init=True, num_hidden=10, dropout=False, batch_norm=True)
gradient_visualizer = GradientVisualizer(deepnet,epochs)
trainer.train(deepnet,gradient_visualizer,epochs=epochs)
show_static(gradient_visualizer)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Linear layer 0',
              'type': 'scatter',
              'uid': 'e0c62a40-545a-4db8-8807-311bd45e11f7',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 1',
              'type': 'scatter',
              'uid': 'a611cc48-1400-4fa8-93fc-f3ca742b7c25',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 2',
              'type': 'scatter',
              'uid': '61678210-e3a0-4107-aa74-504dd11e2789',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 3',
              'type': 'scatter',
              'uid': '4f1dda71-6990-4649-9d7d-9b7d0a5bb1d2',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Linear layer 4',
              'type': 'sca

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.35)'},
              'mode': 'lines',
              'name': 'Epoch 1',
              'type': 'scatter',
              'uid': '6c2cd560-4074-4595-8892-e5b5ed01afcd',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.5499999999999999)'},
              'mode': 'lines',
              'name': 'Epoch 2',
              'type': 'scatter',
              'uid': 'd530aabf-0606-46bc-8e70-600b8bd44c8e',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '0',
              'line': {'color': 'rgba(99, 110, 250,0.75)'},
              'mode': 'lines',
              'name': 'Epoch 3',
              'type': 'scatter',
              'uid': '08b3c777-93b5-4910-a49d-d08416836a4d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'legendgroup': '5',
          

FigureWidget({
    'data': [{'line': {'color': 'rgb(99, 110, 250)'},
              'mode': 'lines',
              'name': 'Linear layer 0',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'bb9d64c3-413a-490d-b3cd-243e5f626fe2',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(239, 85, 59)'},
              'mode': 'lines',
              'name': 'Linear layer 1',
              'showlegend': True,
              'type': 'scatter',
              'uid': 'fedde788-a4c3-496f-a7ce-e9e04161eb6d',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(0, 204, 150)'},
              'mode': 'lines',
              'name': 'Linear layer 2',
              'showlegend': True,
              'type': 'scatter',
              'uid': '38e76259-a8fa-4078-a5d2-8c6895c47102',
              'xaxis': 'x',
              'yaxis': 'y'},
             {'line': {'color': 'rgb(171, 99, 250)'},
   

epoch     1, batch   100, loss 0.741
epoch     1, batch   200, loss 0.322
epoch     1, batch   300, loss 0.269
epoch     1, batch   400, loss 0.231
Accuracy of the network on the 10000 test images: 94.94%
epoch     2, batch   100, loss 0.175
epoch     2, batch   200, loss 0.173
epoch     2, batch   300, loss 0.163
epoch     2, batch   400, loss 0.173
Accuracy of the network on the 10000 test images: 96.08%
epoch     3, batch   100, loss 0.128
epoch     3, batch   200, loss 0.142
epoch     3, batch   300, loss 0.128
epoch     3, batch   400, loss 0.124
Accuracy of the network on the 10000 test images: 96.83%


In [18]:
#Gradient comparison: 10 hidden layers, with vs. without batch norm
#(text version of the GradientVisualizer plots: gradients of each linear layer on the first batch, before training)
def gradient_stats(net, lr=0.05):
    torch.manual_seed(0)
    inputs, labels = next(iter(trainer.trainloader))
    net.train()
    net.zero_grad()
    loss = nn.CrossEntropyLoss()(net(inputs), labels)
    loss.backward()
    grad_stds, ratios = [], []
    for layer in net.layers:
        grad_stds.append(layer.weight.grad.std().item())
        ratios.append((lr * layer.weight.grad.std() / layer.weight.std()).item())
    return grad_stds, ratios

torch.manual_seed(42)
no_bn = DeepNet(glorot_init=True, num_hidden=10, dropout=False, batch_norm=False)
torch.manual_seed(42)
with_bn = DeepNet(glorot_init=True, num_hidden=10, dropout=False, batch_norm=True)

no_bn_grads, no_bn_ratios = gradient_stats(no_bn)
bn_grads, bn_ratios = gradient_stats(with_bn)

print(f"{'layer':>5} | {'grad std (no BN)':>16} | {'grad std (BN)':>13} | {'ratio (no BN)':>13} | {'ratio (BN)':>10}")
for i in range(len(no_bn_grads)):
    print(f"{i:>5} | {no_bn_grads[i]:>16.2e} | {bn_grads[i]:>13.2e} | {no_bn_ratios[i]:>13.2e} | {bn_ratios[i]:>10.2e}")

layer | grad std (no BN) | grad std (BN) | ratio (no BN) | ratio (BN)
    0 |         1.75e-04 |      4.71e-02 |      1.79e-04 |   4.81e-02
    1 |         2.21e-04 |      5.57e-02 |      8.94e-05 |   2.25e-02
    2 |         2.25e-04 |      4.23e-02 |      9.09e-05 |   1.71e-02
    3 |         2.02e-04 |      3.38e-02 |      7.95e-05 |   1.33e-02
    4 |         1.85e-04 |      2.75e-02 |      7.26e-05 |   1.08e-02
    5 |         2.00e-04 |      2.22e-02 |      8.00e-05 |   8.86e-03
    6 |         1.90e-04 |      1.86e-02 |      7.59e-05 |   7.42e-03
    7 |         1.55e-04 |      1.58e-02 |      6.11e-05 |   6.20e-03
    8 |         1.81e-04 |      1.37e-02 |      7.18e-05 |   5.43e-03
    9 |         1.41e-04 |      1.13e-02 |      5.55e-05 |   4.46e-03
   10 |         3.25e-04 |      3.34e-02 |      9.83e-05 |   1.01e-02


# Task 7: Convolutional network

In [19]:
class ConvNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            # block 1: (B, 1, 28, 28) -> (B, 32, 14, 14)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # block 2: (B, 32, 14, 14) -> (B, 64, 7, 7)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            # (B, 64, 7, 7) -> (B, 64 * 7 * 7) -> (B, 10)
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x: Tensor) -> Tensor:
        """
        Input: shape (B, 1, 28, 28)
        Output: shape (B, num_classes=10)
        """
        x = self.features(x)
        x = self.classifier(x)
        return x

In [20]:
#Training the convnet for 5 epochs
#(no GradientVisualizer: it only tracks linear layers)
epochs = 5
convnet = ConvNet()
trainer.train(convnet, None, epochs=epochs)

epoch     1, batch   100, loss 0.249
epoch     1, batch   200, loss 0.074
epoch     1, batch   300, loss 0.063
epoch     1, batch   400, loss 0.053
Accuracy of the network on the 10000 test images: 99.02%
epoch     2, batch   100, loss 0.036
epoch     2, batch   200, loss 0.036
epoch     2, batch   300, loss 0.038
epoch     2, batch   400, loss 0.033
Accuracy of the network on the 10000 test images: 99.16%
epoch     3, batch   100, loss 0.030
epoch     3, batch   200, loss 0.024
epoch     3, batch   300, loss 0.026
epoch     3, batch   400, loss 0.028
Accuracy of the network on the 10000 test images: 99.19%
epoch     4, batch   100, loss 0.020
epoch     4, batch   200, loss 0.023
epoch     4, batch   300, loss 0.020
epoch     4, batch   400, loss 0.020
Accuracy of the network on the 10000 test images: 99.35%
epoch     5, batch   100, loss 0.016
epoch     5, batch   200, loss 0.016
epoch     5, batch   300, loss 0.019
epoch     5, batch   400, loss 0.017
Accuracy of the network on the 1